# Session 9: AI Agents — Concepts and Patterns

This notebook covers:
1. **Agentic RAG** — the bridge from Session 8
2. **LLM vs. Agent** — what changes when you add a control loop
3. **Workflow patterns** — prompt chaining, routing, parallelization, evaluator-optimizer
4. **MCP** — the universal wiring layer for agent tools

We build a small FAISS index inline so everything is self-contained.

In [ ]:
import os
import sys
import numpy as np
import faiss
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Setup complete")

## 1. Agentic RAG — The Bridge from Session 8

In Session 8 we built **single-shot RAG**:
```
query → embed → retrieve → generate → done
```

**Agentic RAG** adds a loop around that pipeline. But there are two very different ways
to implement that loop — and the difference is the central lesson of this session.

**Version 1 (this section):** The programmer writes the loop. We decide when to check
completeness, when to rewrite the query, when to stop. The LLM answers questions,
but the control flow is entirely ours.

**Version 2 (notebook 02):** We give the LLM a `retrieve` tool and let it run its own loop.
The LLM decides when to search, what to search for, and when it has enough to answer.

Version 1 is a **workflow**. Version 2 is an **agent**.
We build Version 1 first so the difference is concrete when we get to Version 2.

We start by building a small in-memory FAISS index over 10 sample texts.

In [ ]:
SAMPLE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next in a pipeline.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def embed_texts(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data], dtype=np.float32)

def embed_query(query: str) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=[query])
    return np.array(response.data[0].embedding, dtype=np.float32)

embeddings = embed_texts(SAMPLE_TEXTS)
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
print(f"FAISS index ready: {faiss_index.ntotal} vectors, dimension {embeddings.shape[1]}")

### The Agentic RAG Loop

Four helper functions, each doing one job:
- `retrieve()` — embed the query and find nearest chunks in FAISS
- `generate()` — call the LLM with retrieved context
- `is_complete()` — ask the LLM if the answer is sufficient
- `rewrite_query()` — ask the LLM what to search for next

In [ ]:
def retrieve(query: str, top_k: int = 2) -> list[str]:
    """Find the top-k most relevant chunks for a query."""
    q_vec = embed_query(query).reshape(1, -1)
    _, indices = faiss_index.search(q_vec, top_k)
    return [SAMPLE_TEXTS[i] for i in indices[0]]

def generate(question: str, context_chunks: list[str]) -> str:
    """Generate an answer grounded in the retrieved context."""
    context = "\n\n".join(context_chunks)
    return client.responses.create(
        model="gpt-4o-mini",
        instructions=(
            "Answer using only the provided context. "
            "If context is insufficient, say exactly what information is missing."
        ),
        input=f"Context:\n{context}\n\nQuestion: {question}"
    ).output_text

def is_complete(question: str, answer: str) -> bool:
    """Ask the LLM whether the answer fully addresses the original question."""
    reply = client.responses.create(
        model="gpt-4o-mini",
        instructions="Reply only 'yes' or 'no'. Does this answer fully address the question?",
        input=f"Question: {question}\n\nAnswer: {answer}"
    ).output_text.strip().lower()
    return reply.startswith("yes")

def rewrite_query(original_question: str, partial_answer: str) -> str:
    """Generate a better search query to find what's still missing."""
    # Simplification: always rewrites from the original question + current partial answer.
    # A production loop would also pass the current query to avoid regenerating the same rewrite.
    return client.responses.create(
        model="gpt-4o-mini",
        instructions="Write a short search query (under 10 words) to find the missing information.",
        input=f"Original question: {original_question}\nPartial answer: {partial_answer}"
    ).output_text.strip()

In [ ]:
def agentic_rag(question: str, max_iterations: int = 3) -> str:
    """
    Agentic RAG control loop:
      1. Retrieve relevant chunks for the current query
      2. Generate an answer from accumulated context
      3. If complete → return. If not → rewrite query and loop.
    """
    query = question
    accumulated_context: list[str] = []

    print(f"Question: {question}\n{'='*60}")

    for i in range(max_iterations):
        print(f"\n[Iteration {i+1}] Searching for: {query}")
        new_chunks = [c for c in retrieve(query) if c not in accumulated_context]
        accumulated_context.extend(new_chunks)
        print(f"  {len(new_chunks)} new chunk(s) retrieved (total: {len(accumulated_context)})")

        answer = generate(question, accumulated_context)
        preview = answer[:120] + ("..." if len(answer) > 120 else "")
        print(f"  Answer: {preview}")

        if is_complete(question, answer):
            print(f"\n✓ Complete after {i+1} iteration(s)")
            return answer

        query = rewrite_query(question, answer)
        print(f"  → Rewriting query: {query}")

    print(f"\n⚠ Reached max iterations ({max_iterations})")
    return answer

answer = agentic_rag(
    "How does FAISS support semantic search, and what similarity metric does it use?"
)
print(f"\nFinal answer:\n{answer}")

### What the programmer still controls

Look at `agentic_rag()` and ask: *who is making each decision?*

| Decision | Who makes it? |
|---|---|
| How many times to try | Programmer (`max_iterations = 3`) |
| Whether to check completeness | Programmer (calls `is_complete()` every iteration) |
| Whether to rewrite the query | Programmer (always calls `rewrite_query()` if not complete) |
| When to stop | Programmer (first `is_complete()` → True, or max reached) |

The LLM is answering questions inside each step — but the **control flow is entirely ours**.
This is a workflow that *simulates* agentic behaviour, not a true agent.

**A true agent:** give the LLM a `retrieve` tool and let `Runner` manage the loop.
The LLM then decides *if* to retrieve, *what* to search for, and *when* it has enough.
We build that in notebook 02.